# PassLLM Targeted: эволюция промпта на Kaggle

**Targeted-атака**: по старому паролю (Old password) из train.json угадываем новый (password).

- **Данные**: соревнование password-guessing — train.json (формат: `{"Knowledge": {"Old password": "..."}, "password": "..."}`), адаптер **126_csdn_disQwen0.5B**.
- **Оценка**: cracked rate по **train.json** (доля точных совпадений угаданного пароля с полем `password`).
- **Ollama** + qwen3:8b для мутаций промпта; PassLLM (Qwen2.5-0.5B + 126_csdn) для оценки.
- Базовая модель и веса: `Qwen/Qwen2.5-0.5B-Instruct` + `/kaggle/input/competitions/password-guessing/126_csdn_disQwen0.5B/126_csdn_disQwen0.5B/`.

!pip install -q openevolve transformers peft accelerate pyyaml flask nest_asyncio

## Ollama + qwen3:8b

## Запуск эволюции (2 итерации)

In [ ]:
# Cell 0: installs + imports

!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

%pip uninstall -y numpy scipy scikit-learn transformers tokenizers torch peft datasets matplotlib pandas tqdm tensorboard tabulate protobuf torchvision torchaudio
%pip install --no-cache-dir --force-reinstall \
    "numpy==1.26.4" \
    "scipy==1.13.1" \
    "scikit-learn==1.5.2" \
    "torch==2.6.0" \
    "transformers==4.47.0" \
    "tokenizers==0.21.0" \
    "datasets==3.5.0" \
    "matplotlib==3.10.0" \
    "pandas==2.2.3" \
    "peft==0.14.0" \
    "tqdm==4.66.5" \
    "tensorboard==2.19.0" \
    "tabulate" \
    "protobuf<6"

%pip install -q openevolve openai accelerate pyyaml flask nest_asyncio "huggingface_hub>=0.30.0" ollama

!git clone https://github.com/Bananiy03/PassLLM.git || true
!which ollama

import os
import sys
import json
import math
import time
import shutil
import threading
import subprocess
import urllib.request
import multiprocessing as mp
from pathlib import Path

import nest_asyncio
import pandas as pd
import torch

from huggingface_hub import snapshot_download

nest_asyncio.apply()

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Python:", sys.version)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("torch:", torch.__version__)


In [ ]:
# Restaring kernel to update numpy and torch 
import os
os._exit(0)

In [2]:
from huggingface_hub import snapshot_download

model_dir = snapshot_download(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct",
    local_dir="/kaggle/working/models/Qwen2.5-0.5B-Instruct",
    local_dir_use_symlinks=False,
)

print("model_dir =", model_dir)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:986: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

...

.gitattributes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

model_dir = /kaggle/working/models/Qwen2.5-0.5B-Instruct


In [3]:
# Cell 1: setup

import os
import json
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/input")
assert ON_KAGGLE, "Этот ноутбук рассчитан на Kaggle"

COMPETITION_DIR = Path("/kaggle/input/competitions/password-guessing")
WORK_DIR = Path("/kaggle/working")
EXPERIMENT_DIR = WORK_DIR / "openevolve_passllm"
PASSLLM_DIR = WORK_DIR / "PassLLM"
QWEN_DIR = WORK_DIR / "models" / "Qwen2.5-0.5B-Instruct"

TRAIN_PATH = COMPETITION_DIR / "train.json"
TEST_PATH = COMPETITION_DIR / "test.json"
ADAPTER_PATH = COMPETITION_DIR / "126_csdn_disQwen0.5B" / "126_csdn_disQwen0.5B"

INITIAL_PROMPT = (
    "As a targeted password guessing model, your task is to utilize "
    "the provided account information to guess the password."
)

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

ENV = {
    "PASSLLM_TRAIN": str(TRAIN_PATH),
    "PASSLLM_ADAPTER_126_CSDN": str(ADAPTER_PATH),
    "PASSLLM_BASE_MODEL": str(QWEN_DIR),
    "PASSLLM_EVAL_DEVICE": "cuda:0",
    "PASSLLM_EVAL_MAX_SAMPLES": "300",
    "PASSLLM_EVAL_TOPK": "100",
    "PASSLLM_EVAL_BATCH_SIZE": "256",
    "PASSLLM_EVAL_EOS_THRESHOLD": "0.01",
    "PASSLLM_EVAL_BEAM_WIDTH_LIST_JSON": json.dumps([48, 1024, 256, 256, 256, 256, 256, 256]),
    "TOKENIZERS_PARALLELISM": "false",
    "OPENAI_API_KEY": "sk-dummy",
    "OPENAI_BASE_URL": "http://127.0.0.1:11434/v1",
    "OPENAI_API_BASE": "http://127.0.0.1:11434/v1",
}

for key, value in ENV.items():
    os.environ[key] = value

print("EXPERIMENT_DIR:", EXPERIMENT_DIR)
print("TRAIN_PATH:", TRAIN_PATH)
print("TEST_PATH:", TEST_PATH)
print("ADAPTER_PATH:", ADAPTER_PATH)
print("QWEN_DIR:", QWEN_DIR)


EXPERIMENT_DIR: /kaggle/working/openevolve_passllm
TRAIN_PATH: /kaggle/input/competitions/password-guessing/train.json
TEST_PATH: /kaggle/input/competitions/password-guessing/test.json
ADAPTER_PATH: /kaggle/input/competitions/password-guessing/126_csdn_disQwen0.5B/126_csdn_disQwen0.5B
QWEN_DIR: /kaggle/working/models/Qwen2.5-0.5B-Instruct


In [4]:
# Cell 2: write initial prompt, evaluator, config, launcher

initial_prompt_path = EXPERIMENT_DIR / "initial_prompt.txt"
initial_prompt_path.write_text(INITIAL_PROMPT, encoding="utf-8")

evaluator_code = r'''
import os
import sys
import json
from pathlib import Path
from typing import Dict, Any, List

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


def find_passllm_root() -> Path:
    candidates = [
        Path("/kaggle/working/PassLLM/Available artifacts for USENIX Security 2025 #772-v1"),
        Path.cwd() / "PassLLM" / "Available artifacts for USENIX Security 2025 #772-v1",
    ]
    for candidate in candidates:
        if (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("PassLLM root not found")


PASSLLM_ROOT = find_passllm_root()
if str(PASSLLM_ROOT) not in sys.path:
    sys.path.insert(0, str(PASSLLM_ROOT))

from src.search.search import dynamic_beam_search
from src.utils.tokenize import get_alpha_vocab


DATA_PATH = os.environ["PASSLLM_TRAIN"]
BASE_MODEL = os.environ["PASSLLM_BASE_MODEL"]
ADAPTER_PATH = os.environ["PASSLLM_ADAPTER_126_CSDN"]
EVAL_DEVICE = os.environ.get("PASSLLM_EVAL_DEVICE", "cuda:0")

MAX_SAMPLES = int(os.environ.get("PASSLLM_EVAL_MAX_SAMPLES", "300"))
TOP_K = int(os.environ.get("PASSLLM_EVAL_TOPK", "100"))
BATCH_SIZE = int(os.environ.get("PASSLLM_EVAL_BATCH_SIZE", "256"))
EOS_THRESHOLD = float(os.environ.get("PASSLLM_EVAL_EOS_THRESHOLD", "0.01"))
BEAM_WIDTH_LIST = json.loads(os.environ["PASSLLM_EVAL_BEAM_WIDTH_LIST_JSON"])

_CACHED_MODEL = None
_CACHED_TOKENIZER = None
_CACHED_VOCAB = None
_CACHED_VOCAB_IDS = None
_CACHED_DATA = None


def normalize(x: str) -> str:
    return (x or "").strip().lower()


def prompt_features(prompt: str):
    p = normalize(prompt)
    length = len(prompt)
    bias = 0.0
    if "old password" in p:
        bias += 0.25
    if "strongest clue" in p or "primary clue" in p:
        bias += 0.15
    if "case" in p or "capital" in p:
        bias += 0.10
    if "digit" in p or "year" in p or "suffix" in p:
        bias += 0.15
    if "repeat" in p or "repeated" in p:
        bias += 0.10
    if "symbol" in p or "substitution" in p:
        bias += 0.10
    if "realistic" in p or "likely" in p:
        bias += 0.10
    if "only the password" in p or "output only" in p:
        bias += 0.05
    return length, min(bias, 1.0)


def load_data():
    global _CACHED_DATA
    if _CACHED_DATA is not None:
        return _CACHED_DATA

    with open(DATA_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    if MAX_SAMPLES > 0:
        data = data[:MAX_SAMPLES]

    _CACHED_DATA = data
    return data


def get_device():
    if torch.cuda.is_available() and EVAL_DEVICE.startswith("cuda"):
        return torch.device(EVAL_DEVICE)
    return torch.device("cpu")


def load_model():
    global _CACHED_MODEL, _CACHED_TOKENIZER, _CACHED_VOCAB, _CACHED_VOCAB_IDS

    if _CACHED_MODEL is not None:
        return _CACHED_MODEL, _CACHED_TOKENIZER, _CACHED_VOCAB, _CACHED_VOCAB_IDS

    device = get_device()
    dtype = torch.float16 if device.type == "cuda" else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=dtype,
        device_map=None,
        trust_remote_code=True,
    ).to(device)

    if ADAPTER_PATH:
        model = PeftModel.from_pretrained(model, ADAPTER_PATH, is_trainable=False)
        model = model.merge_and_unload().to(device)

    model.eval()

    pw_vocab = get_alpha_vocab(tokenizer)
    vocab_ids = list(pw_vocab.values())

    _CACHED_MODEL = model
    _CACHED_TOKENIZER = tokenizer
    _CACHED_VOCAB = pw_vocab
    _CACHED_VOCAB_IDS = vocab_ids
    return model, tokenizer, pw_vocab, vocab_ids


def build_input_ids(tokenizer, instruction: str, old_password: str):
    content = json.dumps({"Old password": old_password}, ensure_ascii=False)
    prompt = f"{tokenizer.bos_token or ''}{instruction.strip()}{content}"
    input_ids = tokenizer(prompt, add_special_tokens=False, return_tensors="pt")["input_ids"]
    return input_ids.squeeze(0)


def decode_results(search_results, vocab) -> List[str]:
    decode_dic = {v: k for k, v in vocab.items()}
    guesses = []
    for seq, _log_prob in search_results:
        text = "".join(decode_dic[token.item()] for token in seq[:-1])
        guesses.append(text)
    return guesses


def rank_of_target(guesses: List[str], target: str):
    target_n = normalize(target)
    for i, g in enumerate(guesses, start=1):
        if normalize(g) == target_n:
            return i
    return None


def evaluate(program_path: str) -> Dict[str, Any]:
    try:
        prompt_text = Path(program_path).read_text(encoding="utf-8").strip()
        if not prompt_text:
            return {
                "combined_score": 0.0,
                "hit_at_10": 0.0,
                "hit_at_100": 0.0,
                "mrr_at_100": 0.0,
                "prompt_length": 0,
                "old_password_bias": 0.0,
                "error": "empty prompt",
            }

        data = load_data()
        model, tokenizer, pw_vocab, vocab_ids = load_model()
        device = next(model.parameters()).device

        hit10 = 0
        hit100 = 0
        mrr = 0.0
        seen = 0

        for item in data:
            old_password = (item.get("Knowledge") or {}).get("Old password", "")
            target = item.get("password", "")
            if not old_password or not target:
                continue

            input_ids = build_input_ids(tokenizer, prompt_text, old_password).to(device)

            with torch.inference_mode():
                search_results = dynamic_beam_search(
                    model=model,
                    input_ids=input_ids,
                    batch_size=BATCH_SIZE,
                    beam_width_list=BEAM_WIDTH_LIST,
                    vocab=vocab_ids,
                    eos_threshold=EOS_THRESHOLD,
                    sorted=True,
                )

            guesses = decode_results(search_results[:TOP_K], pw_vocab)
            rank = rank_of_target(guesses, target)

            if rank is not None and rank <= 10:
                hit10 += 1
            if rank is not None and rank <= 100:
                hit100 += 1
                mrr += 1.0 / rank

            seen += 1

        n = max(seen, 1)
        hit10_rate = hit10 / n
        hit100_rate = hit100 / n
        mrr_rate = mrr / n
        prompt_length, old_password_bias = prompt_features(prompt_text)

        combined_score = 0.60 * hit100_rate + 0.25 * hit10_rate + 0.15 * mrr_rate

        return {
            "combined_score": combined_score,
            "hit_at_10": hit10_rate,
            "hit_at_100": hit100_rate,
            "mrr_at_100": mrr_rate,
            "prompt_length": prompt_length,
            "old_password_bias": old_password_bias,
        }

    except Exception as e:
        try:
            prompt_text = Path(program_path).read_text(encoding="utf-8")
            prompt_length, old_password_bias = prompt_features(prompt_text)
        except Exception:
            prompt_length, old_password_bias = 0, 0.0

        return {
            "combined_score": 0.0,
            "hit_at_10": 0.0,
            "hit_at_100": 0.0,
            "mrr_at_100": 0.0,
            "prompt_length": prompt_length,
            "old_password_bias": old_password_bias,
            "error": str(e),
        }
'''

config_yaml = """
max_iterations: 50
checkpoint_interval: 2
log_level: "INFO"

diff_based_evolution: false
language: "text"
max_code_length: 4000

llm:
  api_base: "http://127.0.0.1:11434/v1"
  models:
    - name: "qwen2.5:7b"
      weight: 1.0
  temperature: 1.1
  max_tokens: 2048
  timeout: 180
  retries: 2

prompt:
  num_top_programs: 3
  num_diverse_programs: 2
  include_artifacts: true
  system_message: |
    You are an expert prompt engineer optimizing a prompt for a targeted password guessing model.
    Keep the same interface: the model receives your instruction followed immediately by a JSON object
    like {"Old password": "..."}.
    Improve only the instruction text.
    Optimize for recovering the user's new password from the old password.
    Favor realistic password update behavior: case changes, numeric suffixes, years, repeated characters,
    symbol substitutions, and minimal memorable edits.
    Do not turn this into a trawling password generation prompt.
    Return only the improved instruction text.

database:
  population_size: 40
  archive_size: 40
  num_islands: 5
  feature_dimensions: ["prompt_length", "old_password_bias"]
  feature_bins: 8
  elite_selection_ratio: 0.15
  exploration_ratio: 0.35
  exploitation_ratio: 0.50
  migration_interval: 4
  migration_rate: 0.15

evaluator:
  timeout: 3600
  max_retries: 2
  parallel_evaluations: 1
  cascade_evaluation: false
"""

launcher_code = r'''
import os
import json
import time
import shutil
import threading
import subprocess
import urllib.request
import multiprocessing as mp
from pathlib import Path

def run_ollama_serve():
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "127.0.0.1:11434"
    env["CUDA_VISIBLE_DEVICES"] = "1"
    subprocess.run(["ollama", "serve"], env=env, check=False)

def ensure_ollama():
    if not (os.path.exists("/usr/local/bin/ollama") or shutil.which("ollama")):
        raise RuntimeError("Ollama not installed")

    thread = threading.Thread(target=run_ollama_serve, daemon=True)
    thread.start()
    time.sleep(5)

    subprocess.run(["ollama", "pull", "qwen2.5:7b"], check=False)

    for _ in range(30):
        try:
            r = urllib.request.urlopen("http://127.0.0.1:11434/v1/models", timeout=5)
            if r.status == 200:
                return
        except Exception:
            time.sleep(2)

    raise RuntimeError("Ollama did not start in time")

def main():
    mp.set_start_method("spawn", force=True)

    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    os.environ["OPENAI_API_KEY"] = "sk-dummy"
    os.environ["OPENAI_BASE_URL"] = "http://127.0.0.1:11434/v1"
    os.environ["OPENAI_API_BASE"] = "http://127.0.0.1:11434/v1"

    os.environ["PASSLLM_TRAIN"] = "/kaggle/input/competitions/password-guessing/train.json"
    os.environ["PASSLLM_ADAPTER_126_CSDN"] = "/kaggle/input/competitions/password-guessing/126_csdn_disQwen0.5B/126_csdn_disQwen0.5B"
    os.environ["PASSLLM_BASE_MODEL"] = "/kaggle/working/models/Qwen2.5-0.5B-Instruct"
    os.environ["PASSLLM_EVAL_DEVICE"] = "cuda:0"
    os.environ["PASSLLM_EVAL_MAX_SAMPLES"] = "300"
    os.environ["PASSLLM_EVAL_TOPK"] = "100"
    os.environ["PASSLLM_EVAL_BATCH_SIZE"] = "256"
    os.environ["PASSLLM_EVAL_EOS_THRESHOLD"] = "0.01"
    os.environ["PASSLLM_EVAL_BEAM_WIDTH_LIST_JSON"] = json.dumps([48, 1024, 256, 256, 256, 256, 256, 256])

    experiment_dir = "/kaggle/working/openevolve_passllm"
    config_path = f"{experiment_dir}/config.yaml"
    evaluator_path = f"{experiment_dir}/evaluator.py"
    initial_prompt_path = f"{experiment_dir}/initial_prompt.txt"
    output_dir = f"{experiment_dir}/output"

    ensure_ollama()

    from openevolve.api import run_evolution
    from openevolve.config import load_config

    config = load_config(config_path)

    result = run_evolution(
        initial_program=initial_prompt_path,
        evaluator=evaluator_path,
        config=config,
        iterations=10,
        output_dir=output_dir,
        cleanup=False,
    )

    print("best_score:", result.best_score)
    print("metrics:", result.metrics)

    best_prompt = None
    if hasattr(result, "best_code") and result.best_code:
        best_prompt = result.best_code
    else:
        candidate = Path(output_dir) / "best" / "best_program.txt"
        if candidate.exists():
            best_prompt = candidate.read_text(encoding="utf-8")

    print("\\n=== BEST PROMPT ===\\n")
    print(best_prompt)

if __name__ == "__main__":
    main()
'''

(EXPERIMENT_DIR / "evaluator.py").write_text(evaluator_code.strip(), encoding="utf-8")
(EXPERIMENT_DIR / "config.yaml").write_text(config_yaml.strip(), encoding="utf-8")
(Path("/kaggle/working/run_openevolve_passllm.py")).write_text(launcher_code.strip(), encoding="utf-8")

print("Wrote:", initial_prompt_path)
print("Wrote:", EXPERIMENT_DIR / "evaluator.py")
print("Wrote:", EXPERIMENT_DIR / "config.yaml")
print("Wrote:", "/kaggle/working/run_openevolve_passllm.py")


Wrote: /kaggle/working/openevolve_passllm/initial_prompt.txt
Wrote: /kaggle/working/openevolve_passllm/evaluator.py
Wrote: /kaggle/working/openevolve_passllm/config.yaml
Wrote: /kaggle/working/run_openevolve_passllm.py


In [ ]:
# Cell 3: run evolution
!python /kaggle/working/run_openevolve_passllm.py


]11;?\]11;?\Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIJlhx1a6T/2//hY2EXeFAAk1nblKPGlPBX/SrQ5cA91D

time=2026-04-24T17:54:37.294Z level=INFO source=routes.go:1752 msg="server config" env="map[CUDA_VISIBLE_DEVICES:1 GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https

In [ ]:
import os
from pathlib import Path

model_dir = Path("/v1/models/Qwen2.5-0.5B-Instruct")
print("exists:", model_dir.exists())
if model_dir.exists():
    print("files:", sorted([p.name for p in model_dir.iterdir()])[:20])


In [ ]:
# Cell 4: inference and prompt A/B

import json
import math
import os
import sys
import time
from pathlib import Path

import torch


def find_passllm_root() -> Path:
    candidates = [
        Path("/kaggle/working/PassLLM/Available artifacts for USENIX Security 2025 #772-v1"),
        Path.cwd() / "PassLLM" / "Available artifacts for USENIX Security 2025 #772-v1",
    ]
    for candidate in candidates:
        if (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("PassLLM root not found")


PASSLLM_ROOT = find_passllm_root()
if str(PASSLLM_ROOT) not in sys.path:
    sys.path.insert(0, str(PASSLLM_ROOT))

from src.model.eval import Basic_Config_For_Evaluation, load_model
from src.search.search import dynamic_beam_search

MODEL_NAME = str(QWEN_DIR)
LORA_PATH = str(ADAPTER_PATH)
TRAIN_FILE = str(TRAIN_PATH)
TEST_FILE = str(TEST_PATH)
BEST_PROMPT_FILE = "/kaggle/working/openevolve_passllm/output/best/best_program.txt"
PAGE_BEST_FILE = "/kaggle/input/competitions/password-guessing/submission_promt.txt"

TOP_K = 100
BATCH_SIZE = 512
BEAM_WIDTH_LIST = [48, 1024] + [256] * 8
EOS_THRESHOLD = 0.01

PROMPTS = {
    
}


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def decode_results(search_results, vocab):
    decode_dic = {v: k for k, v in vocab.items()}
    guesses = []
    for seq, log_prob in search_results:
        text = "".join(decode_dic[token.item()] for token in seq[:-1])
        guesses.append({"guess": text, "prob": math.exp(log_prob.item())})
    return guesses


def build_prompt_input_ids(tokenizer, old_password: str, instruction: str) -> torch.Tensor:
    content = json.dumps({"Old password": old_password}, ensure_ascii=False)
    prompt = f"{tokenizer.bos_token or ''}{instruction}{content}"
    input_ids = tokenizer(prompt, add_special_tokens=False, return_tensors="pt")["input_ids"]
    return input_ids.squeeze(0)


def normalize(x: str) -> str:
    return (x or "").strip().lower()


def load_best_prompt(BEST_PROMPT_FILE):
    with open(BEST_PROMPT_FILE, "r", encoding="utf-8") as f:
        return f.read().strip()


def load_passllm_model():
    base_config = Basic_Config_For_Evaluation(
        base_model=MODEL_NAME,
        tokenizer=MODEL_NAME,
        lora_path=LORA_PATH,
        test_path=TRAIN_FILE,
    )
    model, tokenizer, pw_vocab = load_model(base_config)
    model.eval()
    return model, tokenizer, pw_vocab


def run_inference(prompt_text: str, input_file: str, output_file: str):
    model, tokenizer, pw_vocab = load_passllm_model()
    data = load_json(input_file)
    vocab_ids = list(pw_vocab.values())

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    results = []
    print(f"[*] Loaded {len(data)} rows for inference")

    for idx, entry in enumerate(data, start=1):
        old_password = (entry.get("Knowledge") or {}).get("Old password", "")
        if not old_password:
            continue

        input_ids = build_prompt_input_ids(tokenizer, old_password, prompt_text).to(model.device)

        torch.cuda.synchronize()
        st = time.time()

        with torch.inference_mode():
            search_results = dynamic_beam_search(
                model=model,
                input_ids=input_ids,
                batch_size=BATCH_SIZE,
                beam_width_list=BEAM_WIDTH_LIST,
                vocab=vocab_ids,
                eos_threshold=EOS_THRESHOLD,
                sorted=True,
            )

        torch.cuda.synchronize()
        elapsed = time.time() - st

        decoded = decode_results(search_results[:TOP_K], pw_vocab)
        top_guesses = [item["guess"] for item in decoded]

        print(f"[{idx}/{len(data)}] time={elapsed:.2f}s top5={top_guesses[:5]}")

        results.append({
            "Old password": old_password,
            "Guessed_words": top_guesses,
        })

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"[*] Saved: {output_file}")


def evaluate_prompt(name, instruction, model, tokenizer, pw_vocab, data):
    vocab_ids = list(pw_vocab.values())

    hit1 = 0
    hit10 = 0
    hit100 = 0
    mrr = 0.0
    total_time = 0.0
    collapse_123456789 = 0

    print(f"\n=== TEST: {name} ===")
    print(instruction)

    for idx, entry in enumerate(data, start=1):
        old_password = (entry.get("Knowledge") or {}).get("Old password", "")
        target = entry.get("password", "")

        if not old_password or not target:
            continue

        input_ids = build_prompt_input_ids(tokenizer, old_password, instruction).to(model.device)

        torch.cuda.synchronize()
        st = time.time()

        with torch.inference_mode():
            search_results = dynamic_beam_search(
                model=model,
                input_ids=input_ids,
                batch_size=BATCH_SIZE,
                beam_width_list=BEAM_WIDTH_LIST,
                vocab=vocab_ids,
                eos_threshold=EOS_THRESHOLD,
                sorted=True,
            )

        torch.cuda.synchronize()
        elapsed = time.time() - st
        total_time += elapsed

        decoded = decode_results(search_results[:TOP_K], pw_vocab)
        guesses = [item["guess"] for item in decoded]

        if guesses and guesses[0] == "123456789":
            collapse_123456789 += 1

        rank = None
        target_norm = normalize(target)
        for i, guess in enumerate(guesses, start=1):
            if normalize(guess) == target_norm:
                rank = i
                break

        if rank == 1:
            hit1 += 1
        if rank is not None and rank <= 10:
            hit10 += 1
        if rank is not None and rank <= 100:
            hit100 += 1
            mrr += 1.0 / rank

        if idx <= 3:
            print(f"[{idx}] old={old_password}")
            print(f"target={target}")
            print(f"top5={guesses[:5]}")
            print(f"time={elapsed:.2f}s rank={rank}")

    n = len(data)
    return {
        "prompt_name": name,
        "avg_time_sec": total_time / n,
        "hit@1": hit1 / n,
        "hit@10": hit10 / n,
        "hit@100": hit100 / n,
        "mrr@100": mrr / n,
        "top1_is_123456789_rate": collapse_123456789 / n,
    }


def run_prompt_ab_test(max_samples=200):
    model, tokenizer, pw_vocab = load_passllm_model()
    data = load_json(TRAIN_FILE)[:max_samples]

    all_metrics = []
    for name, instruction in PROMPTS.items():
        metrics = evaluate_prompt(name, instruction, model, tokenizer, pw_vocab, data)
        all_metrics.append(metrics)

    all_metrics.sort(key=lambda x: (x["hit@100"], x["mrr@100"], -x["avg_time_sec"]), reverse=True)

    print("\n=== FINAL RANKING ===")
    for i, row in enumerate(all_metrics, start=1):
        print(
            f"{i}. {row['prompt_name']} | "
            f"hit@100={row['hit@100']:.6f} | "
            f"mrr@100={row['mrr@100']:.6f} | "
            f"hit@10={row['hit@10']:.6f} | "
            f"avg_time={row['avg_time_sec']:.2f}s | "
            f"top1=123456789 rate={row['top1_is_123456789_rate']:.6f}"
        )

    with open("/kaggle/working/prompt_ab_results.json", "w", encoding="utf-8") as f:
        json.dump(all_metrics, f, indent=2, ensure_ascii=False)

    print("[*] Saved: /kaggle/working/prompt_ab_results.json")


found_best_prompt = load_best_prompt(BEST_PROMPT_FILE)
page_best_prompt = load_best_prompt()
PROMPTS["evolved_best"] = best_prompt
PROMPTS["page_best"] = page_best_prompt
# 1) A/B test on train
run_prompt_ab_test(max_samples=200)

# 2) Inference on test with best evolved prompt
run_inference(
    prompt_text=best_prompt,
    input_file=TEST_FILE,
    output_file="/kaggle/working/guessed_words.json",
)


In [ ]:
# Cell 5: guessed_words.json -> submission.csv 

import json
import pandas as pd

pred_file = "/kaggle/working/guessed_words.json"
test_file = "/kaggle/input/competitions/password-guessing/test.json"
csv_file = "/kaggle/working/submission.csv"

with open(pred_file, "r", encoding="utf-8") as f:
    preds = json.load(f)

with open(test_file, "r", encoding="utf-8") as f:
    test_data = json.load(f)

rows = []
for idx, test_item in enumerate(test_data):
    guesses = []
    if idx < len(preds):
        guesses = preds[idx].get("Guessed_words", [])[:100]

    guesses = [
        "0" if g is None or str(g).strip().lower() in {"", "null", "none", "nan"} else str(g)
        for g in guesses
    ]
    guesses += ["0"] * (100 - len(guesses))

    row = {"id": idx}
    for j, g in enumerate(guesses, start=1):
        row[f"password{j}"] = g
    rows.append(row)

df = pd.DataFrame(rows).fillna("0")
df.to_csv(csv_file, index=False)


df.head()
